# 🤖 3주차 실습 | AI 기초 강의
## 머신러닝 성능 평가 — 직접 해보기

---

## ✅ 이 실습에서 배울 것

| 단계 | 내용 |
|------|------|
| STEP 0 | 준비 — 필요한 도구 불러오기 |
| STEP 1 | 실제값과 예측값이 뭔지 확인하기 |
| STEP 2 | TP / TN / FP / FN 직접 세어보기 |
| STEP 3 | 정확도(Accuracy) 계산하기 |
| STEP 4 | Precision / Recall / F1 계산하기 |
| STEP 5 | 라이브러리로 한 번에 검증하기 |
| STEP 6 | 🎮 예측값을 내 마음대로 바꿔보기 |

---

## ▶ 실행 방법

> **셀 왼쪽의 ▶ 버튼을 클릭하면 코드가 실행됩니다.**  
> 반드시 **위에서 아래 순서대로** 실행하세요!

| 단축키 | 설명 |
|--------|------|
| `Ctrl + Enter` | 현재 셀 실행 |
| `Shift + Enter` | 현재 셀 실행 후 다음 셀로 이동 |


---
## 📦 STEP 0. 준비 — 필요한 도구 불러오기

> ⚠️ **이 셀을 가장 먼저 실행하세요!** 나머지 셀이 모두 이 셀에 의존합니다.

In [ ]:
# ── 필요한 라이브러리 불러오기 ──────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

# ── 한글 폰트 설정 (그래프에서 한글이 깨지지 않도록) ────────────────
import subprocess
subprocess.run(['apt-get', 'install', '-y', '-q', 'fonts-nanum'], capture_output=True)
matplotlib.font_manager._load_fontmanager(try_read_cache=False)

import matplotlib.font_manager as fm
nanum = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
if nanum:
    plt.rcParams['font.family'] = nanum[0]
else:
    plt.rcParams['font.family'] = 'DejaVu Sans'

plt.rcParams['axes.unicode_minus'] = False

print('✅ 준비 완료! 이제 아래 셀을 순서대로 실행하세요.')

---
## 📊 STEP 1. 실제값과 예측값 확인하기

머신러닝에서 성능을 평가하려면 두 가지가 필요합니다.

| 이름 | 의미 | 예시 |
|------|------|------|
| `actual` | **실제 정답** (우리가 이미 알고 있는 답) | 실제로 합격한 학생 목록 |
| `pred` | **AI가 예측한 값** | AI가 합격할 것이라 예측한 학생 목록 |

- `1` = **합격 (양성)**
- `0` = **불합격 (음성)**

In [ ]:
# ── 데이터 정의 ─────────────────────────────────────────────────────
# 실제 정답 (10명의 학생 데이터)
actual = [1, 0, 1, 0, 1, 1, 0, 0, 1, 0]

# AI가 예측한 값
pred   = [1, 1, 1, 0, 0, 1, 0, 0, 1, 0]

# ── 결과 출력 ────────────────────────────────────────────────────────
print('=' * 52)
print(f'실제값  (actual): {actual}')
print(f'예측값  (pred  ): {pred}')
print('=' * 52)

print()
print('[ 한 명씩 비교해보기 ]')
print(f'{"번호":<5} {"실제값":<8} {"예측값":<8} {"결과"}')
print('-' * 35)

for i, (a, p) in enumerate(zip(actual, pred)):
    result = '✅ 맞음' if a == p else '❌ 틀림'
    actual_label = '합격(1)' if a == 1 else '불합격(0)'
    pred_label   = '합격(1)' if p == 1 else '불합격(0)'
    print(f'{i+1:>2}번   {actual_label:<10} {pred_label:<10} {result}')

---
## 🔢 STEP 2. TP / TN / FP / FN 직접 세어보기

예측 결과를 **4가지 경우**로 나눌 수 있어요.

| 이름 | 뜻 | 상황 | 좋은가? |
|------|-----|------|---------|
| **TP** (True Positive) | 맞게 양성 예측 | 실제 합격 → 합격이라 예측 | ✅ 좋음 |
| **TN** (True Negative) | 맞게 음성 예측 | 실제 불합격 → 불합격이라 예측 | ✅ 좋음 |
| **FP** (False Positive) | 틀리게 양성 예측 | 실제 불합격 → 합격이라 예측 | ⚠️ 오경보 |
| **FN** (False Negative) | 틀리게 음성 예측 | 실제 합격 → 불합격이라 예측 | 🚨 위험! |

> 💡 **외우는 팁**: `True/False` = 맞음/틀림, `Positive/Negative` = 양성예측/음성예측

In [ ]:
# ── TP / TN / FP / FN 계산 (for문으로 직접 세기) ────────────────────
tp, tn, fp, fn = 0, 0, 0, 0

print('[ 한 명씩 분류해보기 ]')
print(f'{"번호":<5} {"실제":<8} {"예측":<8} {"분류"}')
print('-' * 40)

for i, (a, p) in enumerate(zip(actual, pred)):
    if   a == 1 and p == 1:
        tp += 1
        label = 'TP ✅ 맞게 합격 예측'
    elif a == 0 and p == 0:
        tn += 1
        label = 'TN ✅ 맞게 불합격 예측'
    elif a == 0 and p == 1:
        fp += 1
        label = 'FP ⚠️  불합격인데 합격이라 예측'
    else:
        fn += 1
        label = 'FN 🚨 합격인데 불합격이라 예측'

    print(f'{i+1:>2}번   {a}         {p}         {label}')

print()
print('=' * 40)
print(f'  TP (맞게 합격 예측):       {tp}개')
print(f'  TN (맞게 불합격 예측):     {tn}개')
print(f'  FP (불합격인데 합격 예측): {fp}개  ← 오경보')
print(f'  FN (합격인데 불합격 예측): {fn}개  ← 놓침')
print('=' * 40)
print(f'  합계: {tp+tn+fp+fn}개 (전체 학생 수)')

In [ ]:
# ── 혼동행렬 시각화 ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))

# 4칸 색상
colors = [['#4CAF82', '#E05C5C'],
          ['#F9C642', '#5CB8B2']]
labels = [[f'TP\n{tp}명\n(맞게 합격 예측)', f'FN\n{fn}명\n(합격 놓침)'],
          [f'FP\n{fp}명\n(오경보)', f'TN\n{tn}명\n(맞게 불합격 예측)']]

for i in range(2):
    for j in range(2):
        rect = plt.Rectangle((j, 1-i), 1, 1,
                              facecolor=colors[i][j], alpha=0.85,
                              edgecolor='white', linewidth=3)
        ax.add_patch(rect)
        ax.text(j+0.5, 1.5-i, labels[i][j],
                ha='center', va='center',
                fontsize=13, fontweight='bold', color='white')

ax.set_xlim(0, 2)
ax.set_ylim(0, 2)
ax.set_xticks([0.5, 1.5])
ax.set_xticklabels(['합격(1) 예측', '불합격(0) 예측'], fontsize=12)
ax.set_yticks([0.5, 1.5])
ax.set_yticklabels(['실제 불합격(0)', '실제 합격(1)'], fontsize=12)
ax.set_title('혼동행렬 (Confusion Matrix)', fontsize=14, fontweight='bold', pad=15)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.set_xlabel('예측값 →', fontsize=12)

plt.tight_layout()
plt.show()
print(f'  초록색(TP, TN) = 맞게 예측  |  나머지 = 틀리게 예측')

---
## 📈 STEP 3. 정확도(Accuracy) 계산하기

$$\text{Accuracy} = \frac{TP + TN}{\text{전체 수}}$$

> **"전체 중 얼마나 맞혔나?"** 를 나타내는 가장 기본적인 지표입니다.

In [ ]:
# ── Accuracy 계산 ────────────────────────────────────────────────────
total      = len(actual)          # 전체 학생 수
correct    = tp + tn              # 맞게 예측한 수
accuracy   = correct / total      # 정확도

print('[ Accuracy 계산 과정 ]')
print(f'  맞게 예측한 수 = TP + TN = {tp} + {tn} = {correct}')
print(f'  전체 수        = {total}')
print(f'  Accuracy       = {correct} ÷ {total} = {accuracy:.2f}  ({accuracy*100:.0f}%)')
print()
print(f'  👉 {total}명 중 {correct}명을 정확하게 예측했습니다.')

---
## 🔍 STEP 4. Precision / Recall / F1 계산하기

| 지표 | 공식 | 의미 |
|------|------|------|
| **Precision** (정밀도) | TP ÷ (TP + FP) | 합격이라 했을 때, 실제로 맞은 비율 |
| **Recall** (재현율) | TP ÷ (TP + FN) | 실제 합격자 중 놓치지 않은 비율 |
| **F1 Score** | 2×P×R ÷ (P+R) | Precision과 Recall의 균형 점수 |

In [ ]:
# ── Precision / Recall / F1 계산 ─────────────────────────────────────
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print('[ Precision 계산 ]')
print(f'  합격이라 예측한 수 = TP + FP = {tp} + {fp} = {tp+fp}')
print(f'  그 중 실제 합격    = TP      = {tp}')
print(f'  Precision          = {tp} ÷ {tp+fp} = {precision:.2f}  ({precision*100:.0f}%)')
print()
print('[ Recall 계산 ]')
print(f'  실제 합격자 수     = TP + FN = {tp} + {fn} = {tp+fn}')
print(f'  그 중 맞게 예측    = TP      = {tp}')
print(f'  Recall             = {tp} ÷ {tp+fn} = {recall:.2f}  ({recall*100:.0f}%)')
print()
print('[ F1 Score 계산 ]')
print(f'  F1 = 2 × {precision:.2f} × {recall:.2f} ÷ ({precision:.2f} + {recall:.2f}) = {f1:.2f}')
print()
print('=' * 42)
print(f'  Accuracy  (정확도): {accuracy:.2f}  ({accuracy*100:.0f}%)')
print(f'  Precision (정밀도): {precision:.2f}  ({precision*100:.0f}%)')
print(f'  Recall    (재현율): {recall:.2f}  ({recall*100:.0f}%)')
print(f'  F1 Score  (균형):   {f1:.2f}  ({f1*100:.0f}%)')
print('=' * 42)

In [ ]:
# ── 지표 시각화 ──────────────────────────────────────────────────────
labels_viz  = ['Accuracy\n정확도', 'Precision\n정밀도', 'Recall\n재현율', 'F1 Score\n균형점수']
values_viz  = [accuracy, precision, recall, f1]
bar_colors  = ['#4A90D9', '#5CB8B2', '#F07C3A', '#4CAF82']

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(labels_viz, values_viz, color=bar_colors,
              edgecolor='white', linewidth=1.5, width=0.55)

for bar, val in zip(bars, values_viz):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.03,
            f'{val:.2f}', ha='center', va='bottom',
            fontsize=14, fontweight='bold', color='#1A2B3C')

ax.set_ylim(0, 1.2)
ax.set_ylabel('점수 (0 ~ 1)', fontsize=12)
ax.set_title('4가지 평가 지표 비교', fontsize=14, fontweight='bold')
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='0.8 기준선')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## ⚡ STEP 5. sklearn 라이브러리로 한 번에 검증하기

지금까지 직접 계산한 값이 맞는지 **sklearn 라이브러리**로 확인해봅시다.

> 실제 현업에서는 sklearn으로 바로 계산하지만, 오늘은 **공식의 원리를 이해하기 위해** 수작업으로 먼저 해봤어요!

In [ ]:
# ── sklearn으로 계산 ─────────────────────────────────────────────────
sk_acc  = accuracy_score(actual, pred)
sk_prec = precision_score(actual, pred, zero_division=0)
sk_rec  = recall_score(actual, pred,    zero_division=0)
sk_f1   = f1_score(actual, pred,        zero_division=0)
sk_cm   = confusion_matrix(actual, pred, labels=[0, 1])

print('── sklearn 결과 ────────────────────────────')
print(f'  Accuracy:  {sk_acc:.2f}')
print(f'  Precision: {sk_prec:.2f}')
print(f'  Recall:    {sk_rec:.2f}')
print(f'  F1 Score:  {sk_f1:.2f}')
print()
print('  Confusion Matrix (sklearn):')
print(f'  [[TN={sk_cm[0][0]}, FP={sk_cm[0][1]}],   ← 실제 불합격')
print(f'   [FN={sk_cm[1][0]}, TP={sk_cm[1][1]}]]   ← 실제 합격')
print('────────────────────────────────────────────')

# 수작업 결과와 비교
match = (
    round(accuracy,  2) == round(sk_acc,  2) and
    round(precision, 2) == round(sk_prec, 2) and
    round(recall,    2) == round(sk_rec,  2) and
    round(f1,        2) == round(sk_f1,   2)
)
print()
if match:
    print('✅ 수작업 계산과 완전히 일치합니다! 정확하게 구현했어요.')
else:
    print('❌ 수작업 계산과 다릅니다. STEP 3~4 코드를 다시 확인해보세요.')

---
## 🎮 STEP 6. 예측값을 내 마음대로 바꿔보기

아래 `my_pred` 리스트의 값을 **0 또는 1로** 바꿔보고, 지표가 어떻게 달라지는지 확인해보세요!

### 실험 과제

| 과제 | my_pred 값 | 예상 결과 |
|------|-----------|----------|
| 모두 합격 예측 | `[1,1,1,1,1,1,1,1,1,1]` | Recall은 높지만 Precision 낮음 |
| 모두 불합격 예측 | `[0,0,0,0,0,0,0,0,0,0]` | Recall = 0, Precision = 0 |
| 완벽한 예측 | `[1,0,1,0,1,1,0,0,1,0]` | 모든 지표 1.0 |
| 완전히 반대 예측 | `[0,1,0,1,0,0,1,1,0,1]` | 모든 지표 0 |


In [ ]:
# ══════════════════════════════════════════════════════════
# 👇 여기를 바꿔보세요! (0 또는 1만 사용 가능, 10개 유지)
my_pred = [1, 0, 1, 0, 1, 1, 0, 0, 1, 0]  # ← 이 값을 수정하세요
# ══════════════════════════════════════════════════════════

actual_fixed = [1, 0, 1, 0, 1, 1, 0, 0, 1, 0]  # 실제값은 바꾸지 마세요

# 자동 계산
my_tp = sum(a==1 and p==1 for a,p in zip(actual_fixed, my_pred))
my_tn = sum(a==0 and p==0 for a,p in zip(actual_fixed, my_pred))
my_fp = sum(a==0 and p==1 for a,p in zip(actual_fixed, my_pred))
my_fn = sum(a==1 and p==0 for a,p in zip(actual_fixed, my_pred))

my_acc  = accuracy_score( actual_fixed, my_pred)
my_prec = precision_score(actual_fixed, my_pred, zero_division=0)
my_rec  = recall_score(   actual_fixed, my_pred, zero_division=0)
my_f1   = f1_score(       actual_fixed, my_pred, zero_division=0)

print(f'내 예측값: {my_pred}')
print(f'실제  값:  {actual_fixed}')
print()
print(f'  TP={my_tp}  TN={my_tn}  FP={my_fp}  FN={my_fn}')
print(f'  Accuracy:  {my_acc:.2f}  ({my_acc*100:.0f}%)')
print(f'  Precision: {my_prec:.2f}  ({my_prec*100:.0f}%)')
print(f'  Recall:    {my_rec:.2f}  ({my_rec*100:.0f}%)')
print(f'  F1 Score:  {my_f1:.2f}  ({my_f1*100:.0f}%)')

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 왼쪽: 지표 막대 차트
names  = ['Accuracy\n정확도', 'Precision\n정밀도', 'Recall\n재현율', 'F1\n균형']
vals   = [my_acc, my_prec, my_rec, my_f1]
cols   = ['#4A90D9', '#5CB8B2', '#F07C3A', '#4CAF82']
bars = axes[0].bar(names, vals, color=cols, edgecolor='white', width=0.55)
for bar, v in zip(bars, vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.03,
                 f'{v:.2f}', ha='center', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 1.25)
axes[0].set_title('내 예측 결과 지표', fontsize=13, fontweight='bold')
axes[0].axhline(y=0.8, color='gray', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3, axis='y')

# 오른쪽: 혼동행렬
cm_colors = [['#4CAF82', '#E05C5C'], ['#F9C642', '#5CB8B2']]
cm_labels = [[f'TP\n{my_tp}', f'FN\n{my_fn}'], [f'FP\n{my_fp}', f'TN\n{my_tn}']]
for i in range(2):
    for j in range(2):
        axes[1].add_patch(plt.Rectangle((j, 1-i), 1, 1,
                          facecolor=cm_colors[i][j], alpha=0.85,
                          edgecolor='white', linewidth=3))
        axes[1].text(j+0.5, 1.5-i, cm_labels[i][j],
                     ha='center', va='center',
                     fontsize=18, fontweight='bold', color='white')
axes[1].set_xlim(0, 2)
axes[1].set_ylim(0, 2)
axes[1].set_xticks([0.5, 1.5])
axes[1].set_xticklabels(['합격 예측', '불합격 예측'], fontsize=11)
axes[1].set_yticks([0.5, 1.5])
axes[1].set_yticklabels(['실제 불합격', '실제 합격'], fontsize=11)
axes[1].set_title('혼동행렬', fontsize=13, fontweight='bold')
axes[1].xaxis.tick_top()

plt.tight_layout()
plt.show()

---
## ✅ 실습 완료! 오늘 배운 것 정리

| 개념 | 핵심 내용 |
|------|-----------|
| `actual` | 실제 정답 |
| `pred` | AI가 예측한 값 |
| **TP** | 실제 양성 → 양성으로 맞게 예측 ✅ |
| **TN** | 실제 음성 → 음성으로 맞게 예측 ✅ |
| **FP** | 실제 음성 → 양성으로 틀리게 예측 ⚠️ |
| **FN** | 실제 양성 → 음성으로 틀리게 예측 🚨 |
| **Accuracy** | (TP+TN) ÷ 전체 → 전체 정확도 |
| **Precision** | TP ÷ (TP+FP) → 양성 예측의 신뢰도 |
| **Recall** | TP ÷ (TP+FN) → 실제 양성을 놓치지 않는 정도 |
| **F1** | 2×P×R ÷ (P+R) → Precision과 Recall의 균형 |

---

### 🚀 다음 시간 예고
**Decision Tree** — 머신러닝 모델을 직접 만들어봅니다!  
오늘 배운 Feature, Label, 성능 평가 개념이 그대로 이어집니다.

---
*AI 기초 강의 | 3주차 실습 완료* 🎉